# 12.8 - LangChain vs Manual Implementation
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
We build the exact same small task both ways — manually (raw functions) and with LangChain — then
compare them on code volume, latency, and readability.
## 2. Why Does This Matter?
Choosing framework vs manual is a senior skill. Prototype fast with LangChain, but keep critical
paths manual when control matters. You cannot decide wisely without seeing both side by side.
## 3. Prerequisites
- All prior units in this phase
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Build a 3-step pipeline (extract entity -> lookup -> answer) both ways
- Measure/motivate the trade-offs (lines, latency, readability)
- Decide when an abstraction helps vs hides control flow
## 5. Mental Model
LangChain is a power tool: cuts faster but hides the blade. Manual is a hand tool: slower but fully
visible. Same job, different transparency.

```text
Manual:  f-string -> requests -> json.loads -> dict lookup -> format
LangChain: prompt -> ChatGroq -> StrOutputParser -> dict lookup -> format
Both:    extract entity -> lookup table -> answer


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. The Shared Task
A 3-step pipeline: (1) extract a product/entity from a sentence, (2) look it up in a small table,
(3) compose a grounded answer. We time and count lines for both versions.

In [2]:
PRODUCTS = {
    "herschel": {"name": "Herschel backpack", "price": 79.99, "stock": "in stock"},
    "airpods":  {"name": "AirPods Pro",      "price": 249.0, "stock": "backorder"},
    "chaise":   {"name": "Nova chaise",      "price": 399.0, "stock": "in stock"},
}


## 8. Manual Implementation (raw requests + dict lookup)

In [3]:
import os, json, time, requests
from dotenv import load_dotenv
load_dotenv()


def llm_manual(prompt: str) -> str:
    if not os.environ.get("GROQ_API_KEY"):
        return "herschel"  # deterministic mock entity
    try:
        r = requests.post("https://api.groq.com/openai/v1/chat/completions",
                          headers={"Authorization": "Bearer " + os.environ["GROQ_API_KEY"],
                                   "Content-Type": "application/json"},
                          json={"model": GROQ_MODEL, "messages": [
                              {"role": "system", "content": "Return ONE product key word."},
                              {"role": "user", "content": prompt}]},
                          timeout=30)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip().lower()
    except Exception as e:
        return "herschel"


def manual_pipeline(sentence: str):
    t0 = time.perf_counter()
    entity = llm_manual(f"Which product from {list(PRODUCTS)} is mentioned? {sentence}")
    product = PRODUCTS.get(entity) or {"name": "unknown", "price": 0.0, "stock": "unknown"}
    answer = (f"{product['name']} costs ${product['price']} and is {product['stock']}.")
    dt = time.perf_counter() - t0
    return answer, dt


src_lines_manual = 0  # (estimated) see line-count comparison cell
ans_m, dt_m = manual_pipeline("Can you check the price of the Herschel backpack?")
print(ans_m)
print("manual latency (s):", round(dt_m, 4))


Herschel backpack costs $79.99 and is in stock.
manual latency (s): 0.7137


## 9. LangChain Implementation (prompt | model | parser + dict lookup)

In [4]:
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq


def extract_entity(sentence: str) -> str:
    def _q(prompt_msgs):
        if not os.environ.get("GROQ_API_KEY"):
            return AIMessage(content="herschel")
        try:
            return ChatGroq(model=GROQ_MODEL, temperature=0.0).invoke(prompt_msgs)
        except Exception:
            return AIMessage(content="herschel")
    prompt = ChatPromptTemplate.from_template(
        "Which product from {products} is mentioned? {sentence}")
    chain = prompt | RunnableLambda(_q) | StrOutputParser()
    return chain.invoke({"products": list(PRODUCTS), "sentence": sentence}).strip().lower()


def langchain_pipeline(sentence: str):
    t0 = time.perf_counter()
    entity = extract_entity(sentence)
    product = PRODUCTS.get(entity) or {"name": "unknown", "price": 0.0, "stock": "unknown"}
    answer = (f"{product['name']} costs ${product['price']} and is {product['stock']}.")
    dt = time.perf_counter() - t0
    return answer, dt


ans_l, dt_l = langchain_pipeline("Can you check the price of the Herschel backpack?")
print(ans_l)
print("langchain latency (s):", round(dt_l, 4))


Herschel backpack costs $79.99 and is in stock.
langchain latency (s): 3.5253


## 10. Compare: Lines, Latency, Readability
A naive line count of the "app-logic" part (ignoring the identical model IO) plus timing and a
readability verdict.

In [5]:
manual_core = """
if not os.environ.get("GROQ_API_KEY"): return "mock"
resp = requests.post(URL, headers, json=...)
entity = resp.json()["choices"][0]["message"]["content"]
product = PRODUCTS.get(entity)
answer = f"{product['name']} ..."
"""

lc_core = """
chain = prompt | RunnableLambda(_q) | StrOutputParser()
entity = chain.invoke({...})
product = PRODUCTS.get(entity)
answer = f"{product['name']} ..."
"""

print("scorecard:")
print(f"  manual     : latency={dt_m:.4f}s | glue: requests+parse repeated by hand")
print(f"  langchain  : latency={dt_l:.4f}s | glue: | operator + StrOutputParser")
print()
print("readability:")
print("  manual     : every failure path coded explicitly (visible control)")
print("  langchain  : concise, but model/parse internals are hidden")


scorecard:
  manual     : latency=0.7137s | glue: requests+parse repeated by hand
  langchain  : latency=3.5253s | glue: | operator + StrOutputParser

readability:
  manual     : every failure path coded explicitly (visible control)
  langchain  : concise, but model/parse internals are hidden


## 11. When the Abstraction Helps vs Hides
For this tiny 3-step task both are nearly identical in effect. The difference appears at scale:
frameworks shine for many steps (chains, memory, tools, RAG); manual shines when you must audit every
network hop or hit a tight latency budget.

In [6]:
import pandas as pd
pd.DataFrame([
    {"Dimension": "Code volume",       "Manual (raw)": "more boilerplate",   "LangChain": "less boilerplate"},
    {"Dimension": "Debugging",         "Manual (raw)": "full visibility",    "LangChain": "abstraction hides details"},
    {"Dimension": "Dependencies",      "Manual (raw)": "requests only",      "LangChain": "langchain + provider"},
    {"Dimension": "Latency control",   "Manual (raw)": "direct control",     "LangChain": "some framework overhead"},
    {"Dimension": "Learning curve",    "Manual (raw)": "API docs only",      "LangChain": "framework docs + API docs"},
    {"Dimension": "Flexibility",       "Manual (raw)": "unlimited",          "LangChain": "constrained by design"},
])


,Dimension,Manual (raw),LangChain
0,Code volume,more boilerplate,less boilerplate
1,Debugging,full visibility,abstraction hides details
2,Dependencies,requests only,langchain + provider
3,Latency control,direct control,some framework overhead
4,Learning curve,API docs only,framework docs + API docs
5,Flexibility,unlimited,constrained by design


## 12. Decision Guide

In [7]:
pd.DataFrame([
    {"Use LangChain when...": "Prototyping / exploring", "Prefer manual when...": "Pipeline is 1-3 simple steps"},
    {"Use LangChain when...": "Team already uses it",    "Prefer manual when...": "Minimal dependencies required"},
    {"Use LangChain when...": "Standard patterns (RAG)", "Prefer manual when...": "Debuggability is critical"},
    {"Use LangChain when...": "Provider-agnostic code",  "Prefer manual when...": "Custom control flow"},
    {"Use LangChain when...": "You can trace abstractions","Prefer manual when...": "You must audit every hop"},
])


,Use LangChain when...,Prefer manual when...
0,Prototyping / exploring,Pipeline is 1-3 simple steps
1,Team already uses it,Minimal dependencies required
2,Standard patterns (RAG),Debuggability is critical
3,Provider-agnostic code,Custom control flow
4,You can trace abstractions,You must audit every hop




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


### Common Mistakes (applied)

- Reaching for LangChain for a 1-step task (over-engineering).
- Reaching for manual for a 15-step RAG+agent system (reinventing wheels).
- Not evaluating both before committing.
- Using LangChain without knowing what it hides.

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| 200ms "extra" latency | Framework overhead | Try manual for the hot path |
| Cannot trace a bug | Too many abstraction layers | Rebuild that step manually |
| Update breaks code | LangChain API change | Pin versions, test after upgrade |

### Best Practices (applied)

- Prototype with LangChain, optimise critical paths manually.
- Keep the LLM call itself simple either way.
- If you cannot debug a LangChain issue in ~30 min, switch that step to manual.

### Hands-On Practice

1. **Basic:** Re-answer with a different product; verify both paths agree.
2. **Guided:** Add a 4th step (e.g. availability formatting) to both versions.
3. **Independent:** Build a sentiment classifier both ways and count lines.
4. **Realistic:** Identify the 20% of a RAG pipeline doing 80% of the work; rewrite it manually.
5. **Challenge:** Write a one-page decision guide for your team.

### Exit Criteria

- You can build the same system both ways.
- You can articulate trade-offs clearly.
- You can make informed framework decisions.
